# Camada Silver — microdados2025_pnd_arq1

**Objetivo:** aplicar a primeira regra de negócio sobre a tabela Bronze:

1. Manter somente registros em que `tp_pres = 555`;
2. Remover as colunas `ds_vt_esc_obj`, `ds_vt_gab_obj` e `ds_vt_ace_obj`;
3. Remover as colunas técnicas de auditoria da Bronze da camada Silver, quando existirem;
4. Persistir o resultado como tabela Delta na camada `silver`.

**Origem:** `pnd2025.bronze.microdados2025_pnd_arq1`

**Destino:** `pnd2025.silver.microdados2025_pnd_arq1`

---

### Decisão sobre auditoria

O Delta Lake mantém histórico transacional da tabela por meio do transaction log e do `DESCRIBE HISTORY`.
Porém, isso **não substitui automaticamente auditoria por linha**, como `data_ingestao`, `arquivo_origem` ou `id_execucao`.

Neste projeto, a recomendação é:

- **Bronze:** manter as informações técnicas de ingestão/auditoria, caso existam;
- **Silver:** remover essas colunas quando elas não fizerem parte do modelo analítico;
- **Delta History:** utilizar para histórico de operações da tabela, não como substituto de metadados de origem por registro.

As colunas de auditoria não serão inventadas neste notebook. A célula de diagnóstico identifica as colunas existentes para que somente os nomes realmente presentes sejam removidos.

## 1. Configuração

Definimos os nomes da tabela de origem e destino.

A tabela Bronze já existe, portanto não fazemos nenhuma leitura de arquivo nesta etapa.

In [0]:
CATALOG = "pnd2025"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.microdados2025_pnd_arq1"
TARGET_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.microdados2025_pnd_arq01"

print(f"Origem : {SOURCE_TABLE}")
print(f"Destino: {TARGET_TABLE}")

Origem : pnd2025.bronze.microdados2025_pnd_arq1
Destino: pnd2025.silver.microdados2025_pnd_arq01


## 2. Criar o schema Silver

Criamos o schema somente se ainda não existir.

Isso deixa o notebook idempotente: executar novamente não gera erro caso o schema já exista.

In [0]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}
""")

print(f"Schema {CATALOG}.{SILVER_SCHEMA} disponível.")

## 3. Ler a tabela Bronze

A Bronze é a fonte desta transformação.

Neste momento ainda não aplicamos nenhuma regra de negócio.

In [0]:
df_bronze = spark.table(SOURCE_TABLE)

print(f"Registros Bronze: {df_bronze.count():,}")
print(f"Colunas Bronze  : {len(df_bronze.columns)}")

display(df_bronze.limit(10))

Registros Bronze: 1,087,359
Colunas Bronze  : 32


nu_ano,co_grupo,co_municipio_prova,sg_uf_municipio_prova,tp_inscricao_pnd,in_reaplicacao,co_caderno,ds_vt_gab_obj,ds_vt_esc_obj,ds_vt_ace_obj,tp_pres,tp_sit_disc,proficiencia,nt_obj,nt_dis,nt_ger,qt_acertos,co_rs_i1,co_rs_i2,co_rs_i3,co_rs_i4,co_rs_i5,co_rs_i6,co_rs_i7,co_rs_i8,co_rs_i9,_source_file,_source_file_name,_source_file_size,_source_file_modified_at,_ingestion_ts,_batch_id
2025,702,2211308,PI,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,BCDACABCDBCBCADACBCCCCCCCCCCCCBCCCBBDBBCDCACBBBBACBBDDDABAACCBCCCCCBDCABBBBBCACC,00800010011000810011110101100111101101100111110110101110008100000110011001110011,555,555,"0,2624","52,5","7,75","57,5",40,E,A,C,B,A,E,B,D,B,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2302107,CE,1,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,BCDABABCDBCABBBBDABDDCBCAABBBDBBCDCBCAAAACBCCCDCBCBCBADDDBAADCAACADAACCCBABBABBA,11801110100100811101000010101100101100101100010000011111018011100010011110100111,555,555,"0,2624","52,5","7,75","57,5",40,D,C,B,B,D,C,A,D,A,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2302503,CE,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,BCBABACAAAABDDDADABAABABCCBACAACCABACBACCBADDBBABCBBDBCBAAACCBAADABCDCCBBACBCBCD,00801111111100801101101110001100101001000100110101001110018001101010011111100001,555,555,"0,2624","52,5","7,75","57,5",40,D,B,C,B,D,C,B,D,B,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2302602,CE,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,BDBAACCADBBDCADBDBCABBDCCCBDCAACACACBCACDABCBDBDCAADBBCDDABCCBAADCBBABDCBACBCBDC,00811100010101811011100111111100001100100100000001001010018111101100101111101001,555,555,"0,2624","52,5","7,75","57,5",40,E,A,C,C,D,C,B,D,B,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2304103,CE,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,BCCBDBCADDBDDAAADDCABBACCCCDCABCCCAADACDDACCAABDBBABDBBBCAADABACDABADDBAABCBADCC,11801101001101811111110111010001000100110100000100011101018110101011000011000011,555,555,"0,2624","52,5","7,75","57,5",40,E,B,D,B,D,C,C,C,C,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2304400,CE,2,0,1,DBCDCABDDDADCBACAACCCCACADBBBABDBDCDDACACBBACBADDCACAACACADCAADDBBDDDBBACBCADBBC,CABDCABDADCDCABCABCCCCAAADDBDBBBADADAACCCCADBBADBBCCAACADAADAABABACCABAADCCBABCB,00811111010110811011111011010010010101101000011100011111018011001000010100100100,555,555,"0,2624","52,5","7,75","57,5",40,E,A,C,B,A,E,D,D,D,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2304400,CE,2,0,4,DBBCAACDCCCDBCACACDCADABDDBABABACDDCACAACADBDCABCBDCDDACACBDDBBABDBACABDCDDBAACB,BBBBABBDBCCDABDCABDBABABDBDADBBCCBDADBABCCADACAACBBCDCACBBBBDBBACBABCABCCDBBDBCD,10810111001100801001100110110100101101100110100010010011018110101100111110100001,555,555,"0,2624","52,5","7,75","57,5",40,E,D,C,B,D,B,A,E,E,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2305506,CE,1,0,4,DBBCAACDCCCDBCACACDCADABDDBABABA

## 4. Conferir o schema e identificar colunas de auditoria

Antes de remover qualquer coluna técnica, fazemos o diagnóstico.

**Importante:** não vamos assumir nomes de colunas de auditoria que não foram informados.

Se a Bronze possuir, por exemplo, `dt_ingestao`, `id_execucao`, `nm_arquivo_origem` etc.,
elas poderão ser adicionadas à lista abaixo.

Não recomendamos remover colunas pelo prefixo (`dt_`, `id_`, etc.), pois isso poderia excluir
colunas legítimas do negócio.

In [0]:
df_bronze.printSchema()

print("Colunas existentes:")
for coluna in df_bronze.columns:
    print(f" - {coluna}")

root
 |-- nu_ano: integer (nullable = true)
 |-- co_grupo: integer (nullable = true)
 |-- co_municipio_prova: integer (nullable = true)
 |-- sg_uf_municipio_prova: string (nullable = true)
 |-- tp_inscricao_pnd: integer (nullable = true)
 |-- in_reaplicacao: integer (nullable = true)
 |-- co_caderno: integer (nullable = true)
 |-- ds_vt_gab_obj: string (nullable = true)
 |-- ds_vt_esc_obj: string (nullable = true)
 |-- ds_vt_ace_obj: string (nullable = true)
 |-- tp_pres: integer (nullable = true)
 |-- tp_sit_disc: integer (nullable = true)
 |-- proficiencia: string (nullable = true)
 |-- nt_obj: string (nullable = true)
 |-- nt_dis: string (nullable = true)
 |-- nt_ger: string (nullable = true)
 |-- qt_acertos: string (nullable = true)
 |-- co_rs_i1: string (nullable = true)
 |-- co_rs_i2: string (nullable = true)
 |-- co_rs_i3: string (nullable = true)
 |-- co_rs_i4: string (nullable = true)
 |-- co_rs_i5: string (nullable = true)
 |-- co_rs_i6: string (nullable = true)
 |-- co_rs_i7

## 5. Definir as colunas técnicas de auditoria que NÃO devem seguir para Silver

O Delta Lake registra operações da tabela no transaction log, mas isso não significa que
informações de origem por linha sejam recuperáveis automaticamente.

Por isso, mantemos a auditoria na Bronze quando ela for útil para rastreabilidade.

Na Silver, removemos somente as colunas explicitamente informadas abaixo.

**Se a sua Bronze possuir outras colunas técnicas de auditoria, coloque os nomes exatos na lista
`COLUNAS_AUDITORIA_BRONZE` após conferir o schema na célula anterior.**

In [0]:
COLUNAS_AUDITORIA_BRONZE = [
        "_source_file",
        "_source_file_name",
        "_source_file_size",
        "_source_file_modified_at",
        "_ingestion_ts",
        "_batch_id"
]

# Mantém somente nomes que realmente existem na tabela.
COLUNAS_AUDITORIA_REMOVER = [
    coluna
    for coluna in COLUNAS_AUDITORIA_BRONZE
    if coluna in df_bronze.columns
]

print("Colunas de auditoria que serão removidas:")
if COLUNAS_AUDITORIA_REMOVER:
    for coluna in COLUNAS_AUDITORIA_REMOVER:
        print(f" - {coluna}")
else:
    print(" - Nenhuma. A lista está vazia ou os nomes não existem na Bronze.")

Colunas de auditoria que serão removidas:
 - _source_file
 - _source_file_name
 - _source_file_size
 - _source_file_modified_at
 - _ingestion_ts
 - _batch_id


## 6. Aplicar a regra de negócio: `tp_pres = 555`

Esta é a primeira regra de negócio da camada Silver.

Mantemos somente os registros cujo valor de `tp_pres` seja `555`.

A condição é aplicada antes das demais transformações para reduzir o volume que seguirá no pipeline.

In [0]:
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    .filter(F.col("tp_pres") == 555)
)

print(f"Registros após filtro tp_pres = 555: {df_silver.count():,}")

display(df_silver.limit(10))

Registros após filtro tp_pres = 555: 760,106


nu_ano,co_grupo,co_municipio_prova,sg_uf_municipio_prova,tp_inscricao_pnd,in_reaplicacao,co_caderno,ds_vt_gab_obj,ds_vt_esc_obj,ds_vt_ace_obj,tp_pres,tp_sit_disc,proficiencia,nt_obj,nt_dis,nt_ger,qt_acertos,co_rs_i1,co_rs_i2,co_rs_i3,co_rs_i4,co_rs_i5,co_rs_i6,co_rs_i7,co_rs_i8,co_rs_i9,_source_file,_source_file_name,_source_file_size,_source_file_modified_at,_ingestion_ts,_batch_id
2025,702,1100205,RO,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,1100205,RO,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,1200203,AC,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,1501402,PA,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,1702208,TO,2,0,4,DBBCAACDCCCDBCACACDCADABDDBABABACDDCACAACADBDCABCBDCDDACACBDDBBABDBACABDCDDBAACB,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,1702406,TO,2,0,2,CDBADBCADBBBDDDADCCABAACCCCACABCDCABDBDCDDACACBBACBADDCACAACACAADDBBDDDBBACBCADB,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,1705508,TO,2,0,3,CACABACDDBCADABDDBADCCCCAACDBBBDCDDBCDCABDBADDCAACACBBACDDBBDCAACACAACBCADBDDBBA,................................................................................,00800000000000800000000000000000000000000000000000000000008000000000000000000000,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA,dbfs:/Volumes/pnd2025/landing/dados_txt/microdados2025_pnd_arq1.txt,microdados2025_pnd_arq1.txt,371539465,2026-10-07T13:06:37.000Z,2026-10-07T18:35:26.251Z,20261007182812-d7e9a884
2025,702,2208403,PI,2,0,1,DBCDCABDDDADCBACAACCCCACADBBBABDBDCDDACACBBACBADDCACAACACADCAADDBBD

## 7. Remover colunas que não pertencem à Silver

Removemos as três colunas especificadas na regra:

- `ds_vt_esc_obj`
- `ds_vt_gab_obj`
- `ds_vt_ace_obj`

O código verifica se as colunas existem antes do `drop`, evitando erro caso a estrutura da origem
seja alterada.

In [0]:
COLUNAS_NEGOCIO_REMOVER = [
    "ds_vt_esc_obj",
    "ds_vt_gab_obj",
    "ds_vt_ace_obj",
]

COLUNAS_NEGOCIO_REMOVER_EXISTENTES = [
    coluna
    for coluna in COLUNAS_NEGOCIO_REMOVER
    if coluna in df_silver.columns
]

df_silver = df_silver.drop(*COLUNAS_NEGOCIO_REMOVER_EXISTENTES)

print("Colunas removidas:")
for coluna in COLUNAS_NEGOCIO_REMOVER_EXISTENTES:
    print(f" - {coluna}")

Colunas removidas:
 - ds_vt_esc_obj
 - ds_vt_gab_obj
 - ds_vt_ace_obj


## 8. Remover auditoria técnica da Bronze

Aplicamos agora a remoção das colunas de auditoria que foram identificadas na etapa 5.

A auditoria continua preservada na Bronze.

A Silver fica focada nos dados tratados para consumo pelas próximas camadas.

In [0]:
if COLUNAS_AUDITORIA_REMOVER:
    df_silver = df_silver.drop(*COLUNAS_AUDITORIA_REMOVER)

print(f"Colunas finais da Silver: {len(df_silver.columns)}")

Colunas finais da Silver: 23


## 9. Validação da estrutura final

Conferimos:

- quantidade de colunas;
- schema;
- ausência das colunas removidas;
- regra `tp_pres = 555`.

In [0]:
df_silver.printSchema()

# Validação da regra principal
qtd_tp_pres_invalido = (
    df_silver
    .filter(F.col("tp_pres") != 555)
    .count()
)

print(f"Registros com tp_pres diferente de 555: {qtd_tp_pres_invalido}")

assert qtd_tp_pres_invalido == 0, (
    "ERRO: existem registros na Silver com tp_pres diferente de 555."
)

# Validação das colunas removidas
colunas_proibidas = set(COLUNAS_NEGOCIO_REMOVER) & set(df_silver.columns)

assert not colunas_proibidas, (
    f"ERRO: colunas de negócio removidas ainda estão presentes: {colunas_proibidas}"
)

print("Validações básicas concluídas com sucesso.")

root
 |-- nu_ano: integer (nullable = true)
 |-- co_grupo: integer (nullable = true)
 |-- co_municipio_prova: integer (nullable = true)
 |-- sg_uf_municipio_prova: string (nullable = true)
 |-- tp_inscricao_pnd: integer (nullable = true)
 |-- in_reaplicacao: integer (nullable = true)
 |-- co_caderno: integer (nullable = true)
 |-- tp_pres: integer (nullable = true)
 |-- tp_sit_disc: integer (nullable = true)
 |-- proficiencia: string (nullable = true)
 |-- nt_obj: string (nullable = true)
 |-- nt_dis: string (nullable = true)
 |-- nt_ger: string (nullable = true)
 |-- qt_acertos: string (nullable = true)
 |-- co_rs_i1: string (nullable = true)
 |-- co_rs_i2: string (nullable = true)
 |-- co_rs_i3: string (nullable = true)
 |-- co_rs_i4: string (nullable = true)
 |-- co_rs_i5: string (nullable = true)
 |-- co_rs_i6: string (nullable = true)
 |-- co_rs_i7: string (nullable = true)
 |-- co_rs_i8: string (nullable = true)
 |-- co_rs_i9: string (nullable = true)

Registros com tp_pres difer

## 10. Verificar quantidade de registros

Guardamos a quantidade final para conferência antes da gravação.

In [0]:
qtd_silver = df_silver.count()

print(f"Registros finais da Silver: {qtd_silver:,}")

Registros finais da Silver: 760,106


## 11. Gravar a tabela Silver em Delta

Utilizamos `overwrite` nesta primeira construção da camada Silver.

Isso é apropriado para uma primeira versão/reprocessamento completo.

**Não significa que o processo final necessariamente deverá utilizar overwrite.**
Em uma próxima etapa, podemos transformar este pipeline em carga incremental com `MERGE`.

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela Silver gravada com sucesso: {TARGET_TABLE}")

Tabela Silver gravada com sucesso: pnd2025.silver.microdados2025_pnd_arq01


## 12. Conferir a tabela Silver

Leitura da tabela persistida para garantir que o resultado gravado corresponde ao DataFrame tratado.

In [0]:
df_silver_final = spark.table(TARGET_TABLE)

print(f"Registros gravados: {df_silver_final.count():,}")
print(f"Colunas gravadas  : {len(df_silver_final.columns)}")

display(df_silver_final.limit(20))

Registros gravados: 760,106
Colunas gravadas  : 23


nu_ano,co_grupo,co_municipio_prova,sg_uf_municipio_prova,tp_inscricao_pnd,in_reaplicacao,co_caderno,tp_pres,tp_sit_disc,proficiencia,nt_obj,nt_dis,nt_ger,qt_acertos,co_rs_i1,co_rs_i2,co_rs_i3,co_rs_i4,co_rs_i5,co_rs_i6,co_rs_i7,co_rs_i8,co_rs_i9
2025,702,1100205,RO,2,0,3,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,1100205,RO,2,0,3,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,1200203,AC,2,0,3,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,1501402,PA,2,0,2,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,1702208,TO,2,0,4,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,1702406,TO,2,0,2,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,1705508,TO,2,0,3,555,555,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,2208403,PI,2,0,1,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,2211001,PI,2,0,3,555,336,"-2,23714",0,0,0,2,NA,NA,NA,NA,NA,NA,NA,NA,NA
2025,702,2211001,PI,2,0,4,555,333,"-2,46117",0,0,0,0,NA,NA,NA,NA,NA,NA,NA,NA,NA


## 13. Validação SQL

Consulta simples para validar a regra de negócio diretamente sobre a tabela Delta.

In [0]:
%sql
SELECT
    tp_pres,
    COUNT(*) AS quantidade
FROM pnd2025.silver.microdados2025_pnd_arq01
GROUP BY tp_pres
ORDER BY tp_pres;

tp_pres,quantidade
555,760106


## 14. Conferir histórico Delta

O Delta Lake possui histórico transacional da tabela.

Isso é diferente de auditoria por linha: o histórico informa operações realizadas na tabela,
enquanto os metadados de origem/auditoria da Bronze permitem rastrear cada registro até sua origem.

In [0]:
%sql
DESCRIBE HISTORY pnd2025.silver.microdados2025_pnd_arq01;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-07T19:20:56.000Z,74666397540334,leonardo.andrade.bi.ia@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""8689110c-193b-4844-97d4-c443a2210b67""}, statsOnLoad -> true)",null,List(4177900118654436),2a0f3a0a-8011-4b5a-9d16-1142a9e122e4,1007-171539-4gvi31zm-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 760106, numOutputBytes -> 4189024)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


## 15. Resumo da transformação

**Bronze**

`pnd2025.bronze.microdados2025_pnd_arq1`

↓

**Regra de negócio**

`tp_pres = 555`

↓

**Remoção de colunas**

`ds_vt_esc_obj`

`ds_vt_gab_obj`

`ds_vt_ace_obj`

 `_source_file`

 `_source_file_name`

 `_source_file_size`

 `_source_file_modified_at`

 `_ingestion_ts`
 
 `_batch_id`

↓

**Auditoria técnica**

Mantida na Bronze; removida da Silver somente se explicitamente listada.

↓

**Silver**

`pnd2025.silver.microdados2025_pnd_arq1`